# Pipeline completa — dai dati grezzi agli Expert Advisor

Notebook **master** che esegue tutti i 9 moduli in un'unica chiamata `run_pipeline`:

`dati → pulizia (M1) → feature (M2) → pattern (M3) → strategie (M4) → validazione (M5) → report (M9) → EA MQL4/MQL5 (M6)`.

Sostituisci i dati sintetici con i tuoi CSV (es. export MetaTrader) per usarla su dati reali.

In [ ]:
import sys
from pathlib import Path
for c in [Path.cwd(), Path.cwd().parent, Path('/kaggle/working/trading-ai')]:
    if (c / 'trading_ai').exists():
        sys.path.insert(0, str(c)); break

from trading_ai.data_engine import generate_ohlcv
from trading_ai.strategy_generator import RiskParams, Filters
from trading_ai.pipeline import run_pipeline, PipelineConfig

In [ ]:
# --- DATI ---
# In produzione:
#   from trading_ai.data_engine import DataEngine
#   raw = DataEngine().load_csv('datasets/EURUSD_M1.csv')   # i tuoi dati reali
raw = generate_ohlcv(n=300_000, start='2021-01-01', freq='1min', seed=3)

# --- CONFIGURAZIONE ---
cfg = PipelineConfig(
    timeframe='H1', horizon=10, n_clusters=25,
    min_profit_factor=1.1, min_count_test=20,
    exportable_only=True,                       # EA fedeli e compilabili
    risk=RiskParams(sl_atr=2, tp_atr=3, be_atr=1, trail_atr=1.5,
                    risk_per_trade=0.01, max_bars=40, max_trades_per_day=3),
    filters=Filters(use_trend=True, min_adx=15),
    validate=True, make_reports=True, export_ea=True,
)

result = run_pipeline(raw, cfg)

In [ ]:
print('Pattern candidati :', len(result.patterns))
print('Strategie         :', len(result.strategies))
print('Strategie robuste :', len(result.robust_strategies))
print('Report generati   :', len(result.reports))
print('EA esportati      :', len(result.ea_files))
if result.validation_table is not None and len(result.validation_table):
    display(result.validation_table)

## Nota sui dati sintetici

Su dati **random walk** è normale e corretto che **0 strategie** risultino robuste: non esiste alcun pattern reale da scoprire. È esattamente la difesa anti-overfitting della piattaforma. Collega dati di mercato veri per ottenere risultati significativi.

## Piattaforma operativa ✅

Tutti i moduli (1–9) sono implementati, testati (`pytest`, 63 test verdi) e orchestrati. Gli artefatti finiscono nelle cartelle: `datasets/`, `strategies/`, `reports/`, `mql4/`, `mql5/`.